# Path Planning On a Grid :: Lazy A* Algorithm

<center><img src="./img/1.png"/></center>


In this lab, we will implement a *Lazy* variant of the `A*` algorithm. The term "Lazy" generally implies that certain computations are postponed until they are absolutely necessary. Specifically, in this lab, we defer *edge evaluations*.

In the standard `A*`, when a node $ s $ is expanded, all its outgoing edges are evaluated. This means we compute the cost of traversing these edges and use these true edge costs to determine the $ g $-values of the successors of $ s $.

Edge evaluations can be time-consuming in some domains. Determining the true cost of an edge might involve complex validation routines, such as collision checking. Therefore, it's practical to evaluate them *lazily*.

To be more precise, when expanding a node $ s $ and generating its successors, we approximate their $ g $-values as:

$$
g(s') = g(s) + cost_{apr}(s,s'),
$$

where $ cost_{apr} $ represents an easily computed conservative estimate of the true cost, meaning $ cost_{apr} \leq cost_{true} $.

We only compute the true cost of the edge $ (s, s') $ when $ s' $ is selected for expansion. This might result in recalculating $ g(s') $ and re-inserting $ s' $ into the Open set.

In summary, this approach can save computation time, as we'll demonstrate in this lab.

Further reading: [Cohen, Phillips, and Likhachev (2014), *Planning Single-arm Manipulations with n-Arm Robots*](https://www.roboticsproceedings.org/rss10/p33.pdf), which describes lazy edge evaluation in a weighted A* search. This lab uses the same idea without heuristic weighting. For a broader discussion of lazy edge evaluation, see [Dellin and Srinivasa (2016)](https://ojs.aaai.org/index.php/ICAPS/article/view/13788).


In [ ]:
import math
from heapq import heappop, heappush
from random import random, seed
from time import sleep
from typing import Tuple, Iterable, Callable, Type, Dict, Optional

%matplotlib inline

## Grid Map Representation 

In [ ]:
from utils.Map import Map

## Evaluating Edges

The essence of Lazy A* lies in its approach to edge evaluations.

- `compute_cost`: This is the time-consuming function responsible for determining the true cost of an edge, represented as $cost_{true}$.

- `estimate_cost`: A quicker alternative function that provides an approximation or an underestimate of an edge's cost, denoted as $cost_{apr}$.

While the provided implementations of `compute_cost` and `estimate_cost` may seem artificial, they are OK for the purposes of this lab.


In [ ]:
def compute_cost(i1: int, j1: int, i2: int, j2: int):
    """
    Computes the cost of simple moves between cells (i1, j1) and (i2, j2).

    Parameters
    ----------
    i1 : int
        Row number of the first cell in the grid.
    j1 : int
        Column number of the first cell in the grid.
    i2 : int
        Row number of the second cell in the grid.
    j2 : int
        Column number of the second cell in the grid.

    Raises
    ----------
    ValueError
        If trying to compute the cost of a non-supported move.
    """
    sleep(0.001)  # 1ms sleep to emulate difficult and time-consuming computation of the true cost
    d = abs(i1 - i2) + abs(j1 - j2)
    seed(i1 + j1 + i2 + j2)
    if d == 1:  # Cardinal move
        return 1
    elif d == 2:  # Diagonal move
        return math.sqrt(2) 
    else:
        raise ValueError("Trying to compute the cost of a non-supported move!")


def estimate_cost(i1, j1, i2, j2):
    """
    Computes the cost of simple moves between cells (i1, j1) and (i2, j2).

    Parameters
    ----------
    i1 : int
        Row number of the first cell in the grid.
    j1 : int
        Column number of the first cell in the grid.
    i2 : int
        Row number of the second cell in the grid.
    j2 : int
        Column number of the second cell in the grid.

    Raises
    ----------
    ValueError
        If trying to compute the cost of a non-supported move.
    """
    d = abs(i1 - i2) + abs(j1 - j2)
    if d == 1:  # Cardinal move
        return 1 - random() # Random simulates additional cost that was not taken into account by the estimate_cost method
    elif d == 2:  # Diagonal move
        return math.sqrt(2) - random()
    else:
        raise ValueError("Trying to compute the cost of a non-supported move!")

## Search Node

The structure resembles that of the vanilla `A*`, but with an added twist. We introduce an additional flag, `g_is_true_cost`, to indicate whether the $g$-value of the node was determined using the approximated cost or the true cost.


In [ ]:
from utils.Node import BaseNode as Node

## Implementing the Search Tree

In [ ]:
class SearchTreePQD:
    """
    SearchTree using a priority queue for OPEN and a dictionary for CLOSED.
    """

    def __init__(self):
        self._open = []  # Priority queue for nodes in OPEN
        self._closed = {}  # Dictionary for nodes in CLOSED (expanded nodes)
        self._enc_open_duplicates = 0  # Number of duplicates encountered in OPEN

    def __len__(self) -> int:
        """
        Returns the size of the search tree. Useful for assessing the memory
        footprint of the algorithm, especially at the final iteration.
        """
        return len(self._open) + len(self._closed)

    def open_is_empty(self) -> bool:
        """
        Checks if OPEN is empty.
        If true, the main search loop should be interrupted.
        """
        # <YOUR CODE GOES HERE>
        pass

    def add_to_open(self, item: Node):
        """
        Adds a node to the search tree, specifically to OPEN. This node is either
        entirely new or a duplicate of an existing node in OPEN.
        This implementation detects duplicates lazily; thus, nodes are added to
        OPEN without initial duplicate checks.
        """
        # <YOUR CODE GOES HERE>
        pass

    def get_best_node_from_open(self) -> Optional[Node]:
        """
        Retrieves the best node from OPEN, defined by the minimum key.
        This node will then be expanded in the main search loop.

        Duplicates are managed here. If a node has been expanded previously
        (and is in CLOSED), it's skipped and the next best node is considered.

        Returns None if OPEN is empty.
        """
        # <YOUR CODE GOES HERE>
        pass

    def add_to_closed(self, item: Node):
        """
        Adds a node to the CLOSED dictionary.
        """
        # <YOUR CODE GOES HERE>
        pass

    def was_expanded(self, item: Node) -> bool:
        """
        Checks if a node has been previously expanded.
        """
        # <YOUR CODE GOES HERE>
        pass

    @property
    def opened(self):
        return self._open

    @property
    def expanded(self):
        return self._closed.values()

    @property
    def number_of_open_duplicates(self):
        return self._enc_open_duplicates

## A* Algorithm Without Reexpansions


The input for the `A*` algorithm:

The input is:
- map representation
- start/goal cells
- heuristic function $^*$
- a reference to the implementation of the SearchTree

The output is:
- path found flag (`true` or `false`)
- last node of the path (so one can unwind it using the parent-pointers and get the full path)
- the number of steps (iterations of the main loop)
- the number of nodes that compose the search tree at the final iteration of the algorithm (=the size of the resultant search tree)
- Number of transitions that were precisely computed (representing the number of accurate edge evaluations).
- OPEN and CLOSED (as iterable collections of nodes) for further visualization purposes
- Dictionary representing edges' evaluations. Keys are the node coordinates in the format (i1, j1, i2, j2). A value of True means the transition was precisely computed, and False means it was estimated. 

PS: You might also want to display, at the final iteration, the number of OPEN duplicates encountered during the search, as shown below:

```print("During the search, the following number of OPEN duplicates was encountered: ", ast.number_of_open_duplicates) ```





In [ ]:
def diagonal_distance(i1: int, j1: int, i2: int, j2: int) -> float:
    """
    Computes the diagonal (octile) distance between two grid cells.

    Parameters
    ----------
    i1, j1 : int
        Coordinates (i, j) of the first cell on the grid.
    i2, j2 : int
        Coordinates (i, j) of the second cell on the grid.

    Returns
    -------
    float
        The diagonal (octile) distance between the two specified grid cells.
    """
    # <YOUR CODE GOES HERE>
    pass

In [ ]:
def astar(
    task_map: Map,
    start_i: int,
    start_j: int,
    goal_i: int,
    goal_j: int,
    heuristic_func: Callable,
    search_tree: Type[SearchTreePQD],
) -> Tuple[
    bool,
    Optional[Node],
    int,
    int,
    int,
    Optional[Iterable[Node]],
    Optional[Iterable[Node]],
    Dict[Tuple[int, int, int, int], bool],
]:
    """
    Implements the A* search algorithm.

    Parameters
    ----------
    task_map : Map
        The grid or map being searched.
    start_i, start_j : int, int
        Starting coordinates.
    goal_i, goal_j : int, int
        Goal coordinates.
    heuristic_func : Callable
        Heuristic function for estimating the distance from a node to the goal.
    search_tree : Type[SearchTreePQD]
        The search tree to use.

    Returns
    -------
    Tuple[bool, Optional[Node], int, int, Optional[Iterable[Node]], Optional[Iterable[Node]]]
        Tuple containing:
        - A boolean indicating if a path was found.
        - The last node in the found path or None.
        - Number of algorithm iterations.
        - Size of the resultant search tree.
        - Number of transitions that were precisely computed (representing the number of accurate edge evaluations).
        - OPEN set nodes for visualization or None.
        - CLOSED set nodes.
        - Dictionary representing edges' evaluations. Keys are the node coordinates in the format (i1, j1, i2, j2). A value of True means the transition was precisely computed, and False means it was estimated. Useful for comparing A* with LazyA*.
    """
    ast = search_tree()
    steps = 0

    transitions_computed = 0
    EDGES = {}

    start_node = Node(start_i, start_j, g=0, h=heuristic_func(start_i, start_j, goal_i, goal_j))
    ast.add_to_open(start_node)

    # <YOUR CODE GOES HERE>
    pass

## Lazy A* Algorithm

The input and output for the `Lazy A*` algorithm remain consistent with those of the standard `A*`.


In [ ]:
def lazy_astar(
    task_map: Map,
    start_i: int,
    start_j: int,
    goal_i: int,
    goal_j: int,
    heuristic_func: Callable,
    search_tree: Type[SearchTreePQD],
) -> Tuple[
    bool,
    Optional[Node],
    int,
    int,
    int,
    Optional[Iterable[Node]],
    Optional[Iterable[Node]],
    Dict[Tuple[int, int, int, int], bool],
]:
    """
    Implements the A* search algorithm.

    Parameters
    ----------
    task_map : Map
        The grid or map being searched.
    start_i, start_j : int, int
        Starting coordinates.
    goal_i, goal_j : int, int
        Goal coordinates.
    heuristic_func : Callable
        Heuristic function for estimating the distance from a node to the goal.
    search_tree : Type[SearchTreePQD]
        The search tree to use.

    Returns
    -------
    Tuple[bool, Optional[Node], int, int, Optional[Iterable[Node]], Optional[Iterable[Node]]]
        Tuple containing:
        - A boolean indicating if a path was found.
        - The last node in the found path or None.
        - Number of algorithm iterations.
        - Size of the resultant search tree.
        - Number of transitions that were precisely computed (representing the number of accurate edge evaluations).
        - OPEN set nodes for visualization or None.
        - CLOSED set nodes.
        - Dictionary representing edges' evaluations. Keys are the node coordinates in the format (i1, j1, i2, j2). A value of True means the transition was precisely computed, and False means it was estimated. Useful for comparing A* with LazyA*.
    """
    ast = search_tree()
    steps = 0

    transitions_computed = 0
    EDGES = {}

    start_node = Node(start_i, start_j, g=0, h=heuristic_func(start_i, start_j, goal_i, goal_j), g_is_true_cost=True)
    ast.add_to_open(start_node)

    # <YOUR CODE GOES HERE>
    pass

## Validating the results

When you finish implementing search algorithms, you'll naturally want to test them. The following functions will help you in doing just that.

`simple_test` runs the pathfinding algorithm on a single, straightforward instance. Tests operate under the assumption that the search function follows the pattern below:

```
search(task_map, start.i, start.j, goal.i, goal.j, *optional arguments*) -> (path_found, last_state, steps, search_tree_size, transitions_computed, open, closed, edges)
```

These automated tests anticipate that the search function, which is passed as an input, adheres to the following structure:

- `task_map` is the grid
- `start.i, start.j` are the  $(i, j)$ coordinates of the starting cell
- `goal.i, goal.j` are the $(i, j)$ coordinates of the goal cell
- `*optional arguments*` represent additional parameters of the search function, passed with *args. For instance, the heuristic function.
- `path_found` is `True` if a path was found and `False` otherwise
- `last_state` denotes the last state of the path. It returns `None` if a path was not found
- `steps` provides the number of search steps taken
- `search_tree_size` represents the number of nodes that make up the search tree at the final iteration of the algorithm, which equates to the size of the resultant search tree.
- `transitions_computed` -- the number of transitions that were precisely computed
- `open` is an iterable collection of the nodes that form the search frontier (OPEN nodes)
- `closed` is an iterable collection of the expanded nodes (CLOSED)
-  `edges` is a dictionary representing edges' evaluations. Keys are the node coordinates in the format (i1, j1, i2, j2). A value of True means the transition was precisely computed, and False means it was estimated. 



In [ ]:
from utils.lab_test import simple_test, massive_test

## Assertion Checks

These checks verify path costs, legal moves, and the number of true edge evaluations.


In [ ]:
from lazy_astar_review_checks import review_lazy_astar
review_lazy_astar(Map, Node, SearchTreePQD, diagonal_distance, astar, lazy_astar, compute_cost, estimate_cost)


## Now Let's Test the Implementation

Executing the next two cells will highlight the behavioral differences between LazyA* and A*. If implemented correctly, LazyA* should perform significantly fewer edge evaluations (roughly half as many in this test) and operate at a notably faster pace.

Feel free to run additional tests by modifying the second argument of the `simple_test` function.

In [ ]:
%time simple_test(lazy_astar, 5, diagonal_distance, SearchTreePQD)

In [ ]:
%time simple_test(astar, 5, diagonal_distance, SearchTreePQD)

In [ ]:
%time massive_test(astar, diagonal_distance, SearchTreePQD)
# This may take a while

## Questions

1. Why must the estimated edge cost never exceed the true edge cost?
2. Why might Lazy A* put a node back into OPEN after evaluating its incoming edge? Why must this happen before closing the node or accepting it as the goal?
3. Compare A* and Lazy A* in your tests: path length, true edge evaluations, and runtime. How do these measurements explain the difference you observe?
4. When would delaying edge evaluations provide little benefit, or even make the search slower?


## Your answers

Your answer goes here.
